# Chenda Recording Inspector

Run this **before** PGSD to check whether your recordings are usable.

It will:
1. Show the waveform so you can see if there are multiple strikes or background noise
2. Show the spectrum so you can confirm the Chenda fundamental is present
3. Detect and extract the best single strike automatically
4. Save clean isolated WAV files ready for PGSD
5. Give a go / no-go verdict with the reason

## Cell 1 — Setup

In [ ]:
# ── EDIT THIS PATH ────────────────────────────────────────────────────────────
AUDIO_DATASET = "/kaggle/input/chenda-recordings"
# ─────────────────────────────────────────────────────────────────────────────

import os, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.io import wavfile
from scipy.signal import butter, filtfilt, find_peaks, resample_poly
from scipy.fft import rfft, rfftfreq
from math import gcd
warnings.filterwarnings('ignore')

TARGET_FS = 22050
os.makedirs('/kaggle/working/clean_strikes', exist_ok=True)
os.makedirs('/kaggle/working/inspection',   exist_ok=True)

audio_files = sorted([
    os.path.join(AUDIO_DATASET, f)
    for f in os.listdir(AUDIO_DATASET)
    if f.lower().endswith(('.wav','.mp3','.flac','.m4a','.ogg'))
])
print(f'Found {len(audio_files)} files:')
for f in audio_files:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {os.path.basename(f):40s}  {size_kb:8.1f} KB')

## Cell 2 — Loader

In [ ]:
def load_any(path, target_fs=TARGET_FS):
    ext = os.path.splitext(path)[1].lower()
    if ext == '.wav':
        fs, data = wavfile.read(path)
        if   data.dtype == np.int16:  sig = data / 32768.0
        elif data.dtype == np.int32:  sig = data / 2147483648.0
        elif data.dtype == np.uint8:  sig = (data - 128) / 128.0
        else:                          sig = data.astype(float)
    else:
        try:
            import soundfile as sf
            sig, fs = sf.read(path, always_2d=False)
        except:
            import librosa
            sig, fs = librosa.load(path, sr=None, mono=True)

    sig = np.array(sig, dtype=float)
    if sig.ndim == 2:
        sig = sig.mean(axis=1)
    if fs != target_fs:
        g   = gcd(int(target_fs), int(fs))
        sig = resample_poly(sig, target_fs//g, int(fs)//g)
        fs  = target_fs
    sig /= (np.abs(sig).max() + 1e-12)
    return sig.astype(np.float64), int(fs)

print('Loader ready.')

## Cell 3 — Strike detector
Finds all distinct strikes in a recording. Works on both single strikes and performance recordings.

In [ ]:
def find_strikes(signal, fs,
                 min_gap_s=0.5,
                 min_dur_s=0.3,
                 frame_ms=10):
    """
    Find all strike onsets in a signal.
    Returns list of (onset_sample, offset_sample) pairs.
    """
    frame  = int(fs * frame_ms / 1000)
    energy = np.array([
        np.sum(signal[i:i+frame]**2)
        for i in range(0, len(signal)-frame, frame)
    ])
    energy_db = 10 * np.log10(energy + 1e-12)

    # Dynamic threshold: 20 dB below peak
    threshold = energy_db.max() - 20
    above     = energy_db > threshold

    # Find onset regions
    min_gap_frames = int(min_gap_s * 1000 / frame_ms)
    min_dur_frames = int(min_dur_s * 1000 / frame_ms)

    onsets = []
    in_strike = False
    strike_start = 0
    silent_count = 0

    for i, a in enumerate(above):
        if a and not in_strike:
            strike_start = i
            in_strike    = True
            silent_count = 0
        elif not a and in_strike:
            silent_count += 1
            if silent_count > min_gap_frames:
                dur = i - strike_start
                if dur > min_dur_frames:
                    onsets.append((strike_start * frame, i * frame))
                in_strike    = False
                silent_count = 0
        elif a and in_strike:
            silent_count = 0

    if in_strike:
        onsets.append((strike_start * frame, len(signal)))

    return onsets, energy_db, frame


def extract_best_strike(signal, fs, min_dur_s=1.5, target_dur_s=3.0):
    """
    From a recording, extract the single cleanest isolated strike.
    Picks the loudest strike that has at least min_dur_s of clean decay.
    Returns (strike_signal, onset_time_s, quality_score)
    """
    onsets, energy_db, frame = find_strikes(signal, fs)

    if not onsets:
        # No clear strike found — return full signal trimmed
        trim = min(len(signal), int(fs * target_dur_s))
        return signal[:trim], 0.0, 0.0

    best_sig   = None
    best_score = -np.inf
    best_onset = 0.0

    for onset_s, offset_s in onsets:
        dur = (offset_s - onset_s) / fs
        if dur < min_dur_s:
            continue

        # Score = peak amplitude - background noise level before onset
        pre_start = max(0, onset_s - int(fs * 0.1))
        noise_rms = np.sqrt(np.mean(signal[pre_start:onset_s]**2) + 1e-12)
        peak_amp  = np.abs(signal[onset_s:offset_s]).max()
        snr       = 20 * np.log10(peak_amp / noise_rms)

        # Prefer high SNR and longer decay
        score = snr + np.log(dur)
        if score > best_score:
            best_score = score
            best_onset = onset_s / fs
            # Extract with pre-onset and fixed duration
            pre      = max(0, onset_s - int(fs * 0.02))
            end      = min(len(signal), onset_s + int(fs * target_dur_s))
            seg      = signal[pre:end]
            if len(seg) < int(fs * target_dur_s):
                seg = np.pad(seg, (0, int(fs*target_dur_s) - len(seg)))
            best_sig = seg

    if best_sig is None:
        # All strikes too short — take the longest one anyway
        longest = max(onsets, key=lambda x: x[1]-x[0])
        onset_s, offset_s = longest
        pre     = max(0, onset_s - int(fs * 0.02))
        end     = min(len(signal), onset_s + int(fs * target_dur_s))
        seg     = signal[pre:end]
        if len(seg) < int(fs * target_dur_s):
            seg = np.pad(seg, (0, int(fs*target_dur_s) - len(seg)))
        best_sig   = seg
        best_onset = onset_s / fs
        best_score = 0.0

    return best_sig, best_onset, float(best_score)

print('Strike detector ready.')

## Cell 4 — Signal quality checker
Checks: SNR, duration, whether a Chenda fundamental is detectable, whether it is a single strike or a performance.

In [ ]:
def check_signal_quality(signal, fs, filename=''):
    """
    Returns a quality report dict with a go/no-go verdict.
    """
    report = {'file': filename, 'issues': [], 'warnings': []}

    # ── Duration ─────────────────────────────────────────────────────────────
    dur = len(signal) / fs
    report['duration_s'] = round(dur, 2)
    if dur < 1.0:
        report['issues'].append(f'Too short: {dur:.1f}s (need > 1.0s)')
    elif dur < 2.0:
        report['warnings'].append(f'Short recording: {dur:.1f}s (2-4s ideal)')

    # ── SNR estimate ─────────────────────────────────────────────────────────
    # Compare first 50ms (onset) vs last 500ms (noise floor)
    onset_rms = np.sqrt(np.mean(signal[:int(fs*0.05)]**2) + 1e-12)
    noise_rms = np.sqrt(np.mean(signal[-int(fs*0.5):]**2)  + 1e-12)
    snr_db    = 20 * np.log10(onset_rms / noise_rms)
    report['snr_db'] = round(float(snr_db), 1)
    if snr_db < 10:
        report['issues'].append(f'Very low SNR: {snr_db:.1f} dB (need > 20 dB)')
    elif snr_db < 20:
        report['warnings'].append(f'Low SNR: {snr_db:.1f} dB (noisy background)')

    # ── Chenda fundamental detection ─────────────────────────────────────────
    N    = min(len(signal), 65536)
    mag  = np.abs(rfft(signal[:N] * np.hanning(N)))
    freq = rfftfreq(N, 1.0/fs)

    # Look for a strong peak in 80-250 Hz (Chenda range)
    mask_chenda = (freq >= 80) & (freq <= 250)
    mask_total  = (freq >= 20) & (freq <= 8000)
    if mask_chenda.any() and mask_total.any():
        energy_chenda = mag[mask_chenda].max()
        energy_total  = mag[mask_total].max()
        f1_detected   = float(freq[mask_chenda][np.argmax(mag[mask_chenda])])
        chenda_ratio  = energy_chenda / (energy_total + 1e-12)
        report['f1_hz']        = round(f1_detected, 1)
        report['chenda_ratio'] = round(float(chenda_ratio), 3)
        if chenda_ratio < 0.3:
            report['warnings'].append(
                f'Chenda fundamental weak: {chenda_ratio:.2f} '
                f'(may contain other instruments)')
    else:
        report['f1_hz'] = None
        report['issues'].append('No signal in Chenda frequency range')

    # ── Single strike vs performance ─────────────────────────────────────────
    onsets, energy_db, frame = find_strikes(signal, fs)
    report['n_strikes_detected'] = len(onsets)
    if len(onsets) > 3:
        report['issues'].append(
            f'{len(onsets)} strikes detected — this looks like a performance, '
            f'not a single isolated strike')
    elif len(onsets) > 1:
        report['warnings'].append(
            f'{len(onsets)} strikes detected — will extract the best one')

    # ── Energy flatness (detects continuous noise/music) ─────────────────────
    # If the energy is roughly flat throughout, it is music not a drum
    frame_e = int(fs * 0.1)
    frames  = [signal[i:i+frame_e] for i in range(0, len(signal)-frame_e, frame_e)]
    e_vals  = np.array([np.sum(f**2) for f in frames])
    e_vals  = e_vals / (e_vals.max() + 1e-12)
    flatness = float(e_vals.min() / (e_vals.mean() + 1e-12))
    report['energy_flatness'] = round(flatness, 3)
    if flatness > 0.5:
        report['issues'].append(
            f'Energy is too flat (flatness={flatness:.2f}) — '
            f'this looks like continuous music, not an isolated drum strike')

    # ── T60 estimate (quick) ─────────────────────────────────────────────────
    env = np.abs(signal)
    sm  = max(1, int(fs*0.01))
    env = np.convolve(env, np.ones(sm)/sm, mode='same')
    peak_idx = np.argmax(env)
    tail     = env[peak_idx:]
    t_tail   = np.arange(len(tail)) / fs
    env_db   = 20*np.log10(tail/(tail[0]+1e-12)+1e-12)
    # Find where it drops 60 dB
    below60 = np.where(env_db < -60)[0]
    if len(below60) > 0:
        t60_est = float(t_tail[below60[0]])
    else:
        t60_est = float(t_tail[-1])  # didn't reach -60 dB
    report['t60_est_s'] = round(t60_est, 2)
    if t60_est > 10:
        report['issues'].append(
            f'T60 estimate = {t60_est:.1f}s — far too long for a drum '
            f'(Chenda should be 0.5-2.5s). Signal does not decay cleanly.')
    elif t60_est > 3.0:
        report['warnings'].append(
            f'T60 = {t60_est:.1f}s — slightly long, may have reverb')

    # ── Verdict ───────────────────────────────────────────────────────────────
    if len(report['issues']) == 0:
        report['verdict'] = 'GO'
        report['verdict_reason'] = 'Signal looks good for PGSD'
    else:
        report['verdict'] = 'NO-GO'
        report['verdict_reason'] = ' | '.join(report['issues'])

    return report

print('Quality checker ready.')

## Cell 5 — Inspect all recordings

In [ ]:
inspection_results = []
clean_files = []

for path in audio_files:
    fname = os.path.basename(path)
    print(f'\n{"-"*60}')
    print(f'  File: {fname}')
    print(f'{"-"*60}')

    try:
        sig, fs = load_any(path)
    except Exception as e:
        print(f'  Cannot load: {e}')
        continue

    print(f'  Loaded: {len(sig)/fs:.1f}s at {fs} Hz')

    # Quality check
    report = check_signal_quality(sig, fs, fname)
    inspection_results.append(report)

    # Print report
    print(f'  Duration      : {report["duration_s"]}s')
    print(f'  SNR estimate  : {report["snr_db"]} dB')
    print(f'  f1 detected   : {report["f1_hz"]} Hz')
    print(f'  Strikes found : {report["n_strikes_detected"]}')
    print(f'  T60 estimate  : {report["t60_est_s"]}s')
    print(f'  Energy flat.  : {report["energy_flatness"]} (>0.5 = music, not strike)')

    if report['warnings']:
        for w in report['warnings']:
            print(f'  ⚠ WARNING : {w}')
    if report['issues']:
        for iss in report['issues']:
            print(f'  ✗ ISSUE   : {iss}')

    verdict_sym = '✓' if report['verdict'] == 'GO' else '✗'
    print(f'\n  {verdict_sym} VERDICT: {report["verdict"]} — {report["verdict_reason"]}')

    # ── Waveform + spectrum figure ────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 3))
    fig.suptitle(fname, fontsize=11)
    t_arr = np.arange(len(sig)) / fs

    # Waveform with detected strikes
    ax = axes[0]
    ax.plot(t_arr, sig, linewidth=0.4, color='#378ADD')
    onsets, _, _ = find_strikes(sig, fs)
    for on, off in onsets:
        ax.axvspan(on/fs, off/fs, alpha=0.15, color='#1D9E75')
        ax.axvline(on/fs, color='#1D9E75', linewidth=1, linestyle='--')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude')
    ax.set_title(f'Waveform  ({len(onsets)} strikes highlighted)')

    # Spectrum
    ax = axes[1]
    N   = min(len(sig), 65536)
    mag = np.abs(rfft(sig[:N] * np.hanning(N)))
    frq = rfftfreq(N, 1.0/fs)
    ax.semilogy(frq, mag, color='#378ADD', linewidth=0.6)
    # Mark Chenda range
    ax.axvspan(80, 250, alpha=0.1, color='#1D9E75', label='Chenda f1 range')
    if report['f1_hz']:
        ax.axvline(report['f1_hz'], color='#E24B4A', linewidth=1.5,
                   label=f'Detected f1={report["f1_hz"]}Hz')
    ax.set_xlim(0, 2000); ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude'); ax.set_title('Spectrum')
    ax.legend(fontsize=8)

    # Energy envelope (log)
    ax = axes[2]
    env = np.abs(sig)
    sm  = max(1, int(fs*0.01))
    env = np.convolve(env, np.ones(sm)/sm, mode='same')
    env_db = 20*np.log10(env/(env.max()+1e-12)+1e-12)
    ax.plot(t_arr, env_db, linewidth=0.6, color='#378ADD')
    ax.axhline(-60, color='gray', linewidth=0.8, linestyle=':', label='-60 dB')
    ax.axhline(-20, color='#BA7517', linewidth=0.8, linestyle=':', label='-20 dB')
    ax.set_ylim(-90, 5)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Level (dB)')
    ax.set_title(f'Envelope  T60≈{report["t60_est_s"]}s')
    ax.legend(fontsize=8)

    safe = fname.replace(' ','_').replace('(','').replace(')','').replace('.','_')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/inspection/{safe}.png', dpi=130)
    plt.show(); plt.close()

    # ── Extract and save best strike if signal has any promise ────────────────
    if report['verdict'] == 'GO' or report['snr_db'] > 10:
        clean_sig, onset_t, score = extract_best_strike(sig, fs)
        out_name = f'clean_{fname.replace(" ","_")}'
        if not out_name.endswith('.wav'):
            out_name = os.path.splitext(out_name)[0] + '.wav'
        out_path = f'/kaggle/working/clean_strikes/{out_name}'
        s16 = (clean_sig * 32767).clip(-32768, 32767).astype(np.int16)
        wavfile.write(out_path, fs, s16)
        print(f'  → Extracted strike saved: {out_name}  (onset at {onset_t:.2f}s)')
        clean_files.append(out_path)

print(f'\n{"="*60}')
print(f'SUMMARY')
print(f'{"="*60}')
go    = [r for r in inspection_results if r['verdict'] == 'GO']
no_go = [r for r in inspection_results if r['verdict'] != 'GO']
print(f'  GO     : {len(go)}/{len(inspection_results)} recordings')
print(f'  NO-GO  : {len(no_go)}/{len(inspection_results)} recordings')
if no_go:
    print(f'\n  NO-GO files and reasons:')
    for r in no_go:
        print(f'    {r["file"]}: {r["verdict_reason"]}')

## Cell 6 — What your waveform is telling you
Read this to understand the waveform and envelope plots.

In [ ]:
print("""
HOW TO READ YOUR INSPECTION PLOTS
===================================

WAVEFORM (left panel)
─────────────────────
What you WANT to see:
  One tall spike at the start, then quiet decay
  Like this:  |\\___________

What a YouTube performance looks like:
  Continuous fluctuating energy throughout
  Like this:  /\\/\\/\\/\\/\\/\\

SPECTRUM (middle panel)
───────────────────────
What you WANT to see:
  One dominant peak in the green zone (80-250 Hz)
  Then smaller peaks at multiples (overtones)
  Quiet above 1000 Hz

What a performance looks like:
  Energy spread across all frequencies
  Multiple strong peaks everywhere
  No clear dominant fundamental

ENVELOPE (right panel)
───────────────────────
What you WANT to see:
  Drops from 0 dB to -60 dB in about 1-3 seconds
  Smooth straight line on log scale

What a performance looks like:
  Stays near 0 dB for a long time
  T60 estimate >> 5 seconds
  Bumpy, not smooth

KEY NUMBERS TO CHECK
────────────────────
  SNR > 20 dB       → good isolation from background
  T60 < 3.0 s       → drum decay is captured correctly
  f1 in 100-220 Hz  → Chenda fundamental detected
  Flatness < 0.3    → energy drops after strike (not continuous music)
  R² > 0.80         → PGSD fits the signal well (only after PGSD runs)
""")

## Cell 7 — Run PGSD on clean strikes
Only runs on files that passed inspection. Edit `MODEL_DATASET` path first.

In [ ]:
MODEL_DATASET = "/kaggle/input/chenda-pgsd"  # edit if needed

if not clean_files:
    print('No clean strike files extracted. Check NO-GO reasons above.')
else:
    PGSD_PATH    = os.path.join(MODEL_DATASET, 'chenda_pgsd_v3.py')
    SPECTRAL_PATH= os.path.join(MODEL_DATASET, 'chenda_spectral.py')

    pgsd_src = open(PGSD_PATH).read()
    pgsd_src = pgsd_src.replace(
        'exec(open("/mnt/user-data/outputs/chenda_spectral.py")',
        f'exec(open("{SPECTRAL_PATH}")')
    exec(compile(pgsd_src[:pgsd_src.rfind('\nif __name__')], 'pgsd_v3', 'exec'), globals())
    print('PGSD v3 loaded.')

    SHAPES = ['Cylinder', 'Hourglass', 'Oval']
    BASES  = {}
    for shape in SHAPES:
        fr, mo, al, r, th, _ = get_spectral_basis(shape, T=3500.0)
        BASES[shape] = dict(freqs=fr, modes=mo, alphas=al, r=r, th=th)

    print(f'\nRunning PGSD on {len(clean_files)} clean strike(s)...')
    pgsd_results = []

    for cpath in clean_files:
        fname = os.path.basename(cpath)
        print(f'\n  {fname}')
        sig, fs = load_any(cpath)
        t = np.linspace(0, len(sig)/fs, len(sig), endpoint=False)
        damping = ChendaDamping()

        # Shape ID
        r2s = {}
        for shape in SHAPES:
            Phi = build_basis(BASES[shape]['freqs'], BASES[shape]['alphas'], t)
            _, _, _, r2 = pgsd_decompose(sig, Phi)
            r2s[shape] = float(r2)
        best = max(r2s, key=r2s.get)
        print(f'  Shape: {best}  (R² by shape: '
              + ', '.join(f'{s}={v:.3f}' for s,v in r2s.items()) + ')')

        # Tension
        fr_ref = BASES[best]['freqs']
        al_ref = BASES[best]['alphas']
        f_rf, al_rf, c_f, res_f, fit_f, r2h, _ = pgsd_refine(
            sig, fr_ref.copy(), al_ref.copy(), damping, t,
            n_iter=14, lr=0.25, search_hz=32.0, verbose=False)
        T_est   = 3500.0 * (f_rf[0] / fr_ref[0])**2
        r2_final= float(r2h[-1])
        print(f'  f1_ref={fr_ref[0]:.1f}Hz → f1_refined={f_rf[0]:.1f}Hz → T_est={T_est:.0f} N/m  R²={r2_final:.4f}')

        # Skin CI
        T60_ref = damping.T60(2*np.pi*fr_ref[0])
        f1e = float(f_rf[0])
        fny = fs/2
        flo = max(20., f1e*0.82); fhi = min(fny*0.97, f1e*1.18)
        b,a = butter(4, [flo/fny, fhi/fny], btype='band')
        env = np.abs(filtfilt(b, a, sig))
        sm  = max(1, int(fs*0.005))
        env = np.convolve(env, np.ones(sm)/sm, mode='same')
        noise = env.max()/31.6
        above = np.where(env > noise*2)[0]
        i0 = max(int(fs*0.02), above[0])  if len(above)>100 else int(fs*0.02)
        i1 = above[-1]                     if len(above)>100 else int(fs*0.8)
        t_f = t[i0:i1]; e_f = np.log(np.maximum(env[i0:i1],1e-12))
        fin = np.isfinite(e_f)
        alpha_m = max(-np.polyfit(t_f[fin], e_f[fin], 1)[0], 1e-3) if fin.sum()>20 else 1.0
        T60_m = np.log(1000.)/alpha_m
        CI    = min(100., max(0., (T60_ref-T60_m)/T60_ref*100))
        print(f'  T60_ref={T60_ref:.2f}s → T60_meas={T60_m:.2f}s → CI={CI:.1f}')

        pgsd_results.append({
            'file': fname, 'shape': best, 'r2': r2_final,
            'r2_by_shape': r2s, 'T_est': round(T_est,1),
            'f1_hz': round(float(f_rf[0]),2),
            'T60_s': round(T60_m,3), 'CI': round(CI,1)
        })

    print('\n\n' + '='*64)
    print('PGSD RESULTS ON REAL RECORDINGS')
    print('='*64)
    print(f'  {"File":35}  {"Shape":10}  {"T(N/m)":8}  {"CI":5}  {"R²":7}')
    print('  ' + '─'*68)
    for r in pgsd_results:
        print(f'  {r["file"][:33]:35}  {r["shape"]:10}  '
              f'{r["T_est"]:8.0f}  {r["CI"]:5.1f}  {r["r2"]:7.4f}')

    print('\n' + '─'*64)
    print('JSON — paste to Claude:')
    print('─'*64)
    print(json.dumps({'pgsd_real': pgsd_results,
                      'mean_r2': float(np.mean([r['r2'] for r in pgsd_results]))},
                     indent=2))
    print('─'*64)